# Importation

In [ ]:
# Importation des bibliothèques

# Manipulation de données
import pandas as pd
import numpy as np

# Chargement du modèle et du scaler
import joblib
import json

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
modele = joblib.load("modele_logistique.pkl")
scaler = joblib.load("scaler.pkl")

with open("colonnes_modele.json", "r") as f:
    colonnes = json.load(f)

print(" Modèle, scaler et colonnes chargés avec succès.")
print("Variables du modèle :", colonnes)


 Modèle, scaler et colonnes chargés avec succès.
Variables du modèle : ['diagonal', 'height_left', 'height_right', 'margin_low', 'margin_up', 'length']


In [ ]:
# Chargement du fichier billets_production.csv
df_prod = pd.read_csv("billets_production.csv")

# Aperçu du jeu de données
print("Dimensions :", df_prod.shape)
df_prod.head()


Dimensions : (5, 7)


,diagonal,height_left,height_right,margin_low,margin_up,length,id
0,172.09,103.95,103.73,4.39,3.09,113.19,B_1
1,171.52,104.17,104.03,5.27,3.16,111.82,B_2
2,171.78,103.80,103.75,3.81,3.24,113.39,B_3
3,172.02,104.08,103.99,5.57,3.30,111.10,B_4
4,171.79,104.34,104.37,5.00,3.07,111.87,B_5


# Préparation et standardisation des données

In [ ]:
# Sélection des colonnes utiles pour la prédiction
X_prod = df_prod[colonnes]  # 'colonnes' vient du fichier JSON chargé plus tôt

# Standardisation des données avec le même scaler que celui du modèle entraîné
X_prod_scaled = scaler.transform(X_prod)

print("Données prêtes pour la prédiction. Dimensions :", X_prod_scaled.shape)


Données prêtes pour la prédiction. Dimensions : (5, 6)


Les variables du fichier de production sont mises à l’échelle avec le même StandardScaler que celui utilisé lors de l’entraînement.

Cela garantit que les nouvelles données sont comparables à celles sur lesquelles le modèle a appris,
évitant ainsi tout biais lié à des différences d’unités ou d’échelles.

# Prédiction de la nature des billets

In [ ]:
# Prédiction de la classe (vrai ou faux)
y_pred = modele.predict(X_prod_scaled)

# Prédiction de la probabilité d'authenticité
y_proba = modele.predict_proba(X_prod_scaled)[:, 1]

# Ajout des résultats au dataframe initial
df_prod['prediction'] = np.where(y_pred == 1, "Vrai billet", "Faux billet")
df_prod['probabilité_vrai'] = np.round(y_proba, 3)

# Affichage des résultats
df_prod[['id', 'prediction', 'probabilité_vrai']]


,id,prediction,probabilité_vrai
0,B_1,Vrai billet,0.995
1,B_2,Faux billet,0.003
2,B_3,Vrai billet,0.999
3,B_4,Faux billet,0.000
4,B_5,Faux billet,0.011


Le modèle applique la régression logistique entraînée pour classer chaque billet en “Vrai” ou “Faux”.

La probabilité associée indique le niveau de confiance du modèle dans sa prédiction :
plus elle est proche de 1, plus le billet a de chances d’être authentique.
